In [1]:
!pip install hydra-core lightning torchmetrics wandb omegaconf --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 103.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have nu

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/kiethe/regun-code/ReGUn_visualization.png
/kaggle/input/datasets/kiethe/regun-code/run_slurm.sbatch
/kaggle/input/datasets/kiethe/regun-code/.gitignore
/kaggle/input/datasets/kiethe/regun-code/run2_base.py
/kaggle/input/datasets/kiethe/regun-code/mul_env.def
/kaggle/input/datasets/kiethe/regun-code/README.md
/kaggle/input/datasets/kiethe/regun-code/run1_reference.py
/kaggle/input/datasets/kiethe/regun-code/run4_unlearning_heldout.py
/kaggle/input/datasets/kiethe/regun-code/requirements.txt
/kaggle/input/datasets/kiethe/regun-code/run3_unlearning.py
/kaggle/input/datasets/kiethe/regun-code/utils/loss_history.py
/kaggle/input/datasets/kiethe/regun-code/utils/utils.py
/kaggle/input/datasets/kiethe/regun-code/utils/__init__.py
/kaggle/input/datasets/kiethe/regun-code/models/resnet.py
/kaggle/input/datasets/kiethe/regun-code/models/swin.py
/kaggle/input/datasets/kiethe/regun-code/models/__init__.py
/kaggle/input/datasets/kiethe/regun-code/models/vit.py
/kaggle/input/d

In [3]:
import os

os.environ["CACHE_DIR"]   = "/kaggle/working/cache"
os.environ["DATA_DIR"]    = "/kaggle/working/data"
os.environ["OUTPUTS_DIR"] = "/kaggle/working/outputs"

# Tạo thư mục
os.makedirs("/kaggle/working/cache/models", exist_ok=True)
os.makedirs("/kaggle/working/cache/eval",   exist_ok=True)
os.makedirs("/kaggle/working/outputs",      exist_ok=True)
os.makedirs("/kaggle/working/data",         exist_ok=True)

In [4]:
import os
os.chdir("/kaggle/input/datasets/kiethe/regun-code")

In [5]:
common_overrides = [
    "data=cifar10",
    "model=resnet",
    "model.model.num_classes=10",  # ← thêm
    "trainer.precision=16-mixed",
    "logging.offline=true",
]

In [6]:
# Test mode: chạy ít epoch, ít data để verify pipeline không lỗi
TEST_MODE = False

test_overrides = [
    "trainer.max_epochs=2",
    "+trainer.limit_train_batches=5",  # thêm dấu +
    "+trainer.limit_val_batches=2",    # thêm dấu +
    "trainer.precision=16-mixed",
    "logging.offline=true",
] if TEST_MODE else [
    "trainer.precision=16-mixed",
    "logging.offline=true",
]
print("Mode:", "TEST" if TEST_MODE else "FULL")

Mode: FULL


In [7]:
import subprocess, json, glob, os, pandas as pd
import wandb.proto.wandb_internal_pb2 as pb
from wandb.sdk.internal import datastore

def read_wandb_run(run_dir):
    wandb_files = glob.glob(os.path.join(run_dir, "run-*.wandb"))
    if not wandb_files:
        return None
    store = datastore.DataStore()
    store.open_for_scan(wandb_files[0])
    summary = {}
    while True:
        data = store.scan_data()
        if data is None:
            break
        try:
            record = pb.Record()
            record.ParseFromString(data)
            if record.HasField("summary"):
                for item in record.summary.update:
                    summary[item.key] = item.value_json
        except:
            continue
    return {k: v for k, v in summary.items() if not k.startswith("_")}

def print_summary(summary, title="Results"):
    if not summary:
        print("  (no data)")
        return
    s = pd.Series(summary).astype(float)
    # Gom theo prefix (phần trước "/")
    prefixes = sorted(set(k.split("/")[0] for k in s.index if "/" in k))
    print(f"\n{'═'*55}")
    print(f"  {title}")
    print(f"{'═'*55}")
    for prefix in prefixes:
        subset = s[s.index.str.startswith(f"{prefix}/")]
        if subset.empty:
            continue
        print(f"\n  [{prefix}]")
        print(f"  {'─'*50}")
        # Metrics quan trọng lên trước
        priority = ["acc_retain", "acc_forget", "acc_eval", "acc_test",
                    "average_gap", "average_gap_auc", "rmia_auc"]
        printed = set()
        for km in priority:
            key = f"{prefix}/{km}"
            if key in subset.index:
                print(f"  {'★'} {km:<40} {subset[key]:>8.4f}")
                printed.add(key)
        # Còn lại
        rest = subset[~subset.index.isin(printed)]
        for k, v in rest.items():
            metric = k.split("/", 1)[1]
            print(f"    {metric:<42} {v:>8.4f}")
    print(f"{'═'*55}")

for idx in range(1, 5):
    print(f"\n{'='*40}\nReference model {idx}/4\n{'='*40}")
    result = subprocess.run(
        ["python", "run1_reference.py", f"--run-idx={idx}",
         "data=cifar10", "model=resnet", "model.model.num_classes=10","split.forget_frac=0.01", *test_overrides],
        capture_output=False,
    )
    if result.returncode != 0:
        print(f"[ERROR] Reference {idx} failed")
        break
    latest = max(glob.glob("/kaggle/working/outputs/**/wandb/offline-run-*", recursive=True),
                 key=os.path.getmtime)
    summary = read_wandb_run(latest)
    print_summary(summary, title=f"Reference model {idx}/4")


Reference model 1/4


Seed set to 43


[MAIN] Config:
data:
  name: cifar10
  data_dir: ${paths.data_dir}
  download: true
  batch_size: 128
  num_workers: 4
  pin_memory: true
  persistent_workers: true
  prefetch_factor: 2
  debug_subset_size: null
  transforms:
    normalize:
      mean:
      - 0.4914
      - 0.4822
      - 0.4465
      std:
      - 0.247
      - 0.2435
      - 0.2616
    random_crop:
      enabled: true
      size: 32
      padding: 4
    horizontal_flip:
      enabled: true
      p: 0.5
    imagenet_resize:
      enabled: false
      train_size: 224
      eval_resize: 256
      eval_crop: 224
      normalize:
        mean:
        - 0.485
        - 0.456
        - 0.406
        std:
        - 0.229
        - 0.224
        - 0.225
model:
  model:
    name: resnet18
    num_classes: 10
    pretrained: false
    stem: cifar
    freeze_backbone: false
  optim:
    name: sgd
    lr: 0.1
    weight_decay: 0.0005
    momentum: 0.9
    nesterov: true
    betas:
    - 0.9
    - 0.999
  scheduler:
    name: cos

100%|██████████| 170M/170M [58:10<00:00, 48.9kB/s]
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.25.1
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /kaggle/working/outputs/2026-07-15/06-54-31/wandb/offline-run-20260715_075248-qlo6b44u
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supporte


═══════════════════════════════════════════════════════
  Reference model 1/4
═══════════════════════════════════════════════════════
═══════════════════════════════════════════════════════

Reference model 2/4


...


═══════════════════════════════════════════════════════
  Reference model 2/4
═══════════════════════════════════════════════════════
═══════════════════════════════════════════════════════

Reference model 3/4


Seed set to 45
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.25.1
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /kaggle/working/outputs/2026-07-15/08-48-10/wandb/offline-run-20260715_084813-m7khczks
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated m


═══════════════════════════════════════════════════════
  Reference model 3/4
═══════════════════════════════════════════════════════
═══════════════════════════════════════════════════════

Reference model 4/4


Seed set to 46
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.25.1
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /kaggle/working/outputs/2026-07-15/09-16-01/wandb/offline-run-20260715_091605-aglf34em
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated m


═══════════════════════════════════════════════════════
  Reference model 4/4
═══════════════════════════════════════════════════════
═══════════════════════════════════════════════════════


In [8]:
result = subprocess.run(
    ["python", "run2_base.py", "data=cifar10", "model=resnet","split.forget_frac=0.01", "model.model.num_classes=10", *test_overrides],
    capture_output=False,
)
latest = max(glob.glob("/kaggle/working/outputs/**/wandb/offline-run-*", recursive=True),
             key=os.path.getmtime)
summary = read_wandb_run(latest)
print_summary(summary, title="Base training")

Seed set to 42
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.25.1
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /kaggle/working/outputs/2026-07-15/09-43-51/wandb/offline-run-20260715_094355-ucmglsut
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated m


═══════════════════════════════════════════════════════
  Base training
═══════════════════════════════════════════════════════

  [base]
  ──────────────────────────────────────────────────
  ★ acc_retain                                 1.0000
  ★ acc_forget                                 1.0000
  ★ acc_eval                                   0.9438
  ★ acc_test                                   0.9343
  ★ average_gap                                0.0454
  ★ average_gap_auc                            0.0398
  ★ rmia_auc                                   0.6290
    forget_entropy                               0.0060
    forget_entropy_gap_retrained                -0.0747
    divergence_retain                            0.0004
    divergence_eval                              0.0229
    divergence_test                              0.0240
    divergence_forget                            0.0309
    smia_loss_acc                                0.4040
    smia_loss_auc                     

In [9]:
# ── Unlearning loop  seed=42  CIFAR-10 ─────────────────
import os, subprocess, glob, collections, json
import numpy as np

SEED    = 42
METHODS = ["amun", "finetune", "l1sparse"]

all_results = {}

PRIORITY = [
    "acc_retain", "acc_forget", "acc_eval", "acc_test",
    "average_gap", "average_gap_auc",
    "average_gap_eval", "average_gap_eval_auc",
    "rmia_auc", "rmia_eval_auc",
]

for method in METHODS:
    print(f"\n{'='*45}")
    print(f"  Method={method}  Seed={SEED}  [CIFAR-10]")
    print(f"{'='*45}")

    result = subprocess.run(
        [
            "python", "run3_unlearning.py",
            f"unlearn={method}",
            f"seed={SEED}",
            "split.forget_frac=0.01",
            *common_overrides,
            *test_overrides,
        ],
        capture_output=False,
    )
    if result.returncode != 0:
        print(f"[ERROR] method={method} failed, skipping.")
        continue

    all_runs = glob.glob(
        "/kaggle/working/outputs/**/wandb/offline-run-*",
        recursive=True,
    )
    if not all_runs:
        print("[WARN] No wandb runs found.")
        continue
    latest = max(all_runs, key=os.path.getmtime)
    summary = read_wandb_run(latest)
    if summary is None:
        print("[WARN] No wandb summary found.")
        continue

    all_results[method] = {}
    for key, val in summary.items():
        if key.startswith("unlearning/"):
            try:
                all_results[method][key] = float(val)
            except (ValueError, TypeError):
                pass

    # in ngay sau khi method xong
    print(f"\n  [{method.upper()}]")
    print(f"  {'─'*55}")
    printed = set()
    for short in PRIORITY:
        key = f"unlearning/{short}"
        if key not in all_results[method]:
            continue
        print(f"  ★ {short:<42} {all_results[method][key]*100:6.2f} %")
        printed.add(key)
    for key in sorted(all_results[method].keys()):
        if key in printed:
            continue
        short = key.split("/", 1)[1]
        print(f"    {short:<44} {all_results[method][key]*100:6.2f} %")

# ── Lưu results ─────────────────────────────────────────
with open("/kaggle/working/results_summary.json", "w") as f:
    json.dump(all_results, f, indent=2)
print("\nSaved → /kaggle/working/results_summary.json")


  Method=amun  Seed=42  [CIFAR-10]


Seed set to 42
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.25.1
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /kaggle/working/outputs/2026-07-15/10-41-41/wandb/offline-run-20260715_104145-ivf7xx7m
building AMUN adversarial set: 100%|██████████| 4/4 [00:52<00:00, 13.18s/it]
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.p


  [AMUN]
  ───────────────────────────────────────────────────────
  ★ acc_retain                                  99.46 %
  ★ acc_forget                                  85.11 %
  ★ acc_eval                                    92.38 %
  ★ acc_test                                    91.62 %
  ★ average_gap                                 14.22 %
  ★ average_gap_auc                              5.51 %
  ★ average_gap_eval                             6.66 %
  ★ average_gap_eval_auc                         5.65 %
  ★ rmia_auc                                    43.57 %
  ★ rmia_eval_auc                               42.94 %
    average_gap_eval_test                          6.15 %
    average_gap_test                               5.86 %
    divergence_eval                                3.64 %
    divergence_forget                              9.95 %
    divergence_retain                              0.45 %
    divergence_test                                3.64 %
    forget_entropy      

Seed set to 42
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.25.1
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /kaggle/working/outputs/2026-07-15/10-55-02/wandb/offline-run-20260715_105506-vwba17wk
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated m


  [FINETUNE]
  ───────────────────────────────────────────────────────
  ★ acc_retain                                 100.00 %
  ★ acc_forget                                 100.00 %
  ★ acc_eval                                    94.27 %
  ★ acc_test                                    93.42 %
  ★ average_gap                                  6.21 %
  ★ average_gap_auc                              3.87 %
  ★ average_gap_eval                            11.41 %
  ★ average_gap_eval_auc                         3.93 %
  ★ rmia_auc                                    62.44 %
  ★ rmia_eval_auc                               62.53 %
    average_gap_eval_test                          5.30 %
    average_gap_test                               5.18 %
    divergence_eval                                2.34 %
    divergence_forget                              3.08 %
    divergence_retain                              0.04 %
    divergence_test                                2.46 %
    forget_entropy  

Seed set to 42
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.25.1
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /kaggle/working/outputs/2026-07-15/11-04-54/wandb/offline-run-20260715_110458-td0h5f3v
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated m


  [L1SPARSE]
  ───────────────────────────────────────────────────────
  ★ acc_retain                                 100.00 %
  ★ acc_forget                                 100.00 %
  ★ acc_eval                                    94.09 %
  ★ acc_test                                    93.02 %
  ★ average_gap                                  5.33 %
  ★ average_gap_auc                              4.50 %
  ★ average_gap_eval                             7.60 %
  ★ average_gap_eval_auc                         4.61 %
  ★ rmia_auc                                    64.55 %
  ★ rmia_eval_auc                               65.06 %
    average_gap_eval_test                          6.66 %
    average_gap_test                               6.43 %
    divergence_eval                                2.35 %
    divergence_forget                              2.94 %
    divergence_retain                              0.11 %
    divergence_test                                2.52 %
    forget_entropy  

In [10]:
# ── Cell 4: representation/flip analysis (retrain vs unlearned) ──
# Dùng lại SEED, METHODS từ cell 3.
# Yêu cầu: cell 3 phải chạy bằng run4_unlearning_heldout.py (đổi "run3_unlearning.py"
# → "run4_unlearning_heldout.py") để có file state_dict unlearned được lưu.
import os, torch, pandas as pd
from hydra import compose, initialize_config_dir
from evaluation.metrics_representation import (
    compare_models_on_split, summarize_flips, flips_to_dataframe,
    extract_all_splits, save_embeddings,
)
from data import build_datamodule
from models import build_model, load_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for method in METHODS:
    with initialize_config_dir(config_dir=os.path.join(os.getcwd(), "conf"), version_base=None):
        cfg = compose(config_name="config",
                      overrides=[f"unlearn={method}", f"seed={SEED}",
                                 "split.forget_frac=0.01", *common_overrides])

    sd_path = (f"/kaggle/working/cache/models/unlearned_{method}_resnet18_cifar10_"
               f"random_forget0.01_{SEED}.pt")
    if not os.path.exists(sd_path):
        print(f"[skip] no saved unlearned model for {method}"); continue

    dm = build_datamodule(cfg); dm.setup()
    model_retrained = load_model(cfg, cfg.run.retrain_weights)
    model_unlearned = build_model(cfg)
    model_unlearned.load_state_dict(torch.load(sd_path, map_location="cpu"))

    print(f"\n{'='*45}\n  Representation: {method}  (retrain vs unlearned)\n{'='*45}")
    for split, loader in [("forget", dm.forget_eval_dataloader()),
                          ("retain", dm.retain_eval_dataloader()),
                          ("test",   dm.test_dataloader())]:
        flip = compare_models_on_split(model_retrained, model_unlearned, loader, device,
                                       label_a="retrain", label_b="unlearned")
        print(f"[{split}]", summarize_flips(flip))
        if split == "forget":
            df = flips_to_dataframe(flip, split_name=split)
            df.to_csv(f"/kaggle/working/flips_{method}_{split}.csv", index=False)
            display(df[df.flipped].sort_values("embed_l2_dist", ascending=False).head(20))

    emb = extract_all_splits(model_unlearned, dm, device)
    save_embeddings(emb, f"/kaggle/working/embeddings_{method}.npz")
    print(f"Saved embeddings_{method}.npz")

[skip] no saved unlearned model for amun
[skip] no saved unlearned model for finetune
[skip] no saved unlearned model for l1sparse
